# B-03a: SARIMAX baseline (classical statistical model, follows B-03 config)

Fills the gap in the original model roster (D-05: persistence/seasonal-mean -> **ARIMA** -> RF/XGBoost ->
LSTM/TFT) -- ARIMA/SARIMAX was scoped but never actually run in any forecasting notebook. This adds it as a
per-tower classical baseline on the same data/CV/horizons as B-03 (`forecast_features_v2.csv` /
`forecast_daily_v2.csv`, Towers 4/9 main split test 2022-2023, Tower 2 expanding-window folds).

**Exogenous drivers (SARIMAX, not pure univariate ARIMA):** a small, SHAP-informed set rather than B-03's full
~26-34 column feature matrix -- `results/fc_importance_shap_rf.csv` (I-01) ranks `fx_lsu_dens` (livestock
density) as the #1 forecasting driver by a wide margin, followed by wind speed / VPD / USTAR / PPFD. Those five
plus two seasonality proxies (day-of-year sin/cos for daily; is_daytime/is_growing for hourly) keep the
exogenous state space small enough for tractable SARIMAX fitting at this row count. AR lags (`ar_ch4_*`) are
deliberately excluded from the exogenous set -- SARIMAX's own `(p,d,q)` terms already model that
autoregressive structure; including the lagged target as an exogenous regressor would be redundant.

**Methodological departure from B-03 (documented, not a bug):** ARIMA/SARIMAX is a per-series classical model
-- there is no standard panel/hierarchical SARIMAX equivalent to the partial-pooling (D-30) the rest of the
project uses, so this fits **per tower, solo** (no pooling, no tower dummies). **Evaluation departs from the
direct multi-horizon design too**: rather than one model per horizon, SARIMAX is fit ONCE per tower on
2018-2021 training data, then walked forward through the test period (`.append(refit=False)`, a fast
Kalman-filter state update with no re-optimisation) -- at each strided origin, `get_forecast(steps=H)` with
known future exogenous values produces predictions at every evaluation horizon simultaneously, which is
ARIMA's natural and standard multi-step-ahead usage.

In [1]:
from pathlib import Path
import sys, datetime, time, warnings, numpy as np, pandas as pd
sys.path.insert(0, "../../src")
warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from evaluation.metrics import full_metrics
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
HORIZONS={"A":[1,6,12,24,48],"B":[1,3,7,14]}
T2_FOLDS=[("2018-06-30","2018-07-01","2018-12-31"),("2018-12-31","2019-01-01","2019-05-31")]
STRIDE={"A":24,"B":1}  # hourly walk-forward is the expensive part (Kalman re-filter grows with series length);
                       # stride=24 (vs DL's 6) keeps total runtime bounded for a "small experiment"

matv2=pd.read_csv(HOURLY/"forecast_features_v2.csv",low_memory=False)
matv2["Datetime"]=pd.to_datetime(matv2["Datetime"],format="mixed")
dv=pd.read_csv(HOURLY/"forecast_daily_v2.csv",low_memory=False)
dv["Datetime"]=pd.to_datetime(dv["Datetime"],format="mixed")

EXOG_A=["fx_lsu_dens","fx_WS_0_0_1","fx_VPD_0_0_1","fx_USTAR_0_0_1","fx_PPFD_1_1_1","fx_is_daytime","fx_is_growing"]
EXOG_B=["fx_lsu_dens","fx_WS_mean","fx_VPD_mean","fx_USTAR_mean","fx_PPFD_mean","fx_DOY_sin","fx_DOY_cos","fx_is_growing"]
print("Track A exog:",EXOG_A)
print("Track B exog:",EXOG_B)

Track A exog: ['fx_lsu_dens', 'fx_WS_0_0_1', 'fx_VPD_0_0_1', 'fx_USTAR_0_0_1', 'fx_PPFD_1_1_1', 'fx_is_daytime', 'fx_is_growing']
Track B exog: ['fx_lsu_dens', 'fx_WS_mean', 'fx_VPD_mean', 'fx_USTAR_mean', 'fx_PPFD_mean', 'fx_DOY_sin', 'fx_DOY_cos', 'fx_is_growing']


## 1  Helpers -- order search (bounded AIC grid), walk-forward fit/forecast

In [2]:
def rmse(y,p): return float(np.sqrt(mean_squared_error(y,p)))
def metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r2=r2_score(y,p) if (len(y)>1 and np.var(y)>0) else np.nan
    return rmse(y,p), float(mean_absolute_error(y,p)), r2, float(np.mean(p-y))

def prep_series(df, exog_cols):
    y=df["y_gapfilled"].astype(float)
    X=df[exog_cols].astype(float).ffill().bfill()
    return y, X

def search_order(y_tr, X_tr):
    # bounded grid (d=1 fixed -- flux is non-stationary/trending over the multi-year training window;
    # p,q kept small) to keep the SARIMAX MLE refit cost tractable for a "small experiment".
    # Returns the fitted result object directly (not just the order) to avoid a redundant 5th refit.
    best=None
    for p in [1,2]:
        for q in [0,1]:
            try:
                t0=time.time()
                m=SARIMAX(y_tr,exog=X_tr,order=(p,1,q),enforce_stationarity=False,enforce_invertibility=False)
                r=m.fit(disp=False,maxiter=50)
                print(f"    order=({p},1,{q}) aic={r.aic:.1f} ({time.time()-t0:.1f}s)",flush=True)
                if best is None or r.aic<best[0]:
                    best=(r.aic,(p,1,q),r)
            except Exception as e:
                print(f"    order=({p},1,{q}) failed: {e}",flush=True); continue
    return (best[1],best[2]) if best else ((1,1,1),None)

def fit_walk_forward(y, X, fit_end_pos, horizons, order, res, stride):
    """Continue from the already-fitted `res` (from search_order, avoids a redundant refit), then
    walk forward to the end of the series via append(refit=False), forecasting `horizons` ahead at
    each strided origin."""
    if res is None:
        m=SARIMAX(y.iloc[:fit_end_pos+1],exog=X.iloc[:fit_end_pos+1],order=order,enforce_stationarity=False,enforce_invertibility=False)
        res=m.fit(disp=False,maxiter=50)
    n=len(y); H=max(horizons); pos=fit_end_pos; rows=[]
    while pos+stride<n:
        new_end=min(pos+stride,n-1)
        new_y=y.iloc[pos+1:new_end+1]; new_X=X.iloc[pos+1:new_end+1]
        if len(new_y)>0:
            res=res.append(new_y,exog=new_X,refit=False); pos=new_end
        if pos+H<n:
            future_X=X.iloc[pos+1:pos+1+H]
            fc=res.get_forecast(steps=H,exog=future_X)
            pred=fc.predicted_mean.values
            origin_time=y.index[pos]
            for h in horizons:
                rows.append(dict(origin=origin_time,horizon=h,target_time=y.index[pos+h],y_pred=float(pred[h-1])))
    return rows
print("helpers ready")

helpers ready


## 2  Run one track for one tower (main split: fit on <=2021, walk forward through 2022-2023)

In [3]:
def run_main_tower(track, t, df, exog_cols, unit):
    print(f"  -- Track {track} Tower {t}: order search --",flush=True)
    y,X=prep_series(df,exog_cols)
    fit_end=df.index.get_indexer([pd.Timestamp("2021-12-31 23:00:00" if unit=="h" else "2021-12-31")],method="nearest")[0]
    order,res=search_order(y.iloc[:fit_end+1],X.iloc[:fit_end+1])
    print(f"  -- Track {track} Tower {t}: chosen order={order}, walking forward --",flush=True)
    fc_rows=fit_walk_forward(y,X,fit_end,HORIZONS[track],order,res,STRIDE[track])
    FC=pd.DataFrame(fc_rows)
    obs=df["y_observed"].rename("y_true")
    FC=FC.merge(obs,left_on="target_time",right_index=True,how="left").dropna(subset=["y_true"])
    FC=FC[(FC.target_time.dt.year.isin([2022,2023]))]
    rows=[]
    for h in HORIZONS[track]:
        sub=FC[FC.horizon==h]
        if len(sub)<10: continue
        r,mae,r2,mbe=metrics(sub.y_true.values,sub.y_pred.values)
        persist=df.loc[sub.origin,"y_gapfilled"].values
        rp=rmse(sub.y_true.values,persist)
        fm=full_metrics(sub.y_true.values,sub.y_pred.values,y_naive=persist)
        rows.append(dict(track=track,horizon=h,tower=t,model="SARIMAX",order=str(order),
            RMSE=round(r,3),MAE=round(mae,3),R2=(round(r2,3) if np.isfinite(r2) else np.nan),MBE=round(mbe,3),
            n_test=len(sub),skill_persist=round(1-r/rp,3) if rp>0 else np.nan,
            WAPE=round(fm["WAPE"],4) if np.isfinite(fm["WAPE"]) else np.nan,
            MASE=round(fm["MASE"],4) if np.isfinite(fm["MASE"]) else np.nan,
            sMAPE=round(fm["sMAPE"],4) if np.isfinite(fm["sMAPE"]) else np.nan,
            MAPE=round(fm["MAPE"],4) if np.isfinite(fm["MAPE"]) else np.nan,
            MAPE_n_excluded=fm["MAPE_n_excluded"]))
    return rows,order

ALL=[]
TA={t: matv2[matv2.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
TB={t: dv[dv.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
for t in [4,9]:
    r,order=run_main_tower("A",t,TA[t],EXOG_A,"h"); ALL+=r; print(f"Track A Tower {t} order={order} done",flush=True)
for t in [4,9]:
    r,order=run_main_tower("B",t,TB[t],EXOG_B,"D"); ALL+=r; print(f"Track B Tower {t} order={order} done",flush=True)
print("main-split rows so far:",len(ALL))

  -- Track A Tower 4: order search --


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


    order=(1,1,0) aic=519933.8 (7.7s)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


    order=(1,1,1) aic=513948.9 (27.3s)


...

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


    order=(2,1,1) aic=19445.6 (1.4s)


  -- Track B Tower 9: chosen order=(2, 1, 1), walking forward --


Track B Tower 9 order=(2, 1, 1) done


main-split rows so far: 18


## 3  Tower 2 (expanding-window folds, donor-free -- Tower 2's own short series)

In [4]:
def run_t2_folds(track, df, exog_cols, unit):
    y,X=prep_series(df,exog_cols)
    acc=[]
    for cut,s,e in T2_FOLDS:
        fit_end_ts=pd.Timestamp(cut+(" 23:00:00" if unit=="h" else ""))
        if fit_end_ts not in df.index:
            fit_end=df.index.get_indexer([fit_end_ts],method="nearest")[0]
        else:
            fit_end=df.index.get_loc(fit_end_ts)
        y_tr=y.iloc[:fit_end+1]
        if len(y_tr)<50: continue
        print(f"  -- Track {track} Tower 2 fold (cut={cut}): order search --",flush=True)
        order,res=search_order(y_tr,X.iloc[:fit_end+1])
        fc_rows=fit_walk_forward(y,X,fit_end,HORIZONS[track],order,res,STRIDE[track])
        FC=pd.DataFrame(fc_rows)
        if FC.empty: continue
        FC=FC[(FC.target_time>=pd.Timestamp(s))&(FC.target_time<=pd.Timestamp(e))]
        acc.append(FC)
    if not acc: return []
    FC=pd.concat(acc,ignore_index=True)
    obs=df["y_observed"].rename("y_true")
    FC=FC.merge(obs,left_on="target_time",right_index=True,how="left").dropna(subset=["y_true"])
    rows=[]
    for h in HORIZONS[track]:
        sub=FC[FC.horizon==h]
        if len(sub)<5: continue
        r,mae,r2,mbe=metrics(sub.y_true.values,sub.y_pred.values)
        persist=df.loc[sub.origin,"y_gapfilled"].values
        rp=rmse(sub.y_true.values,persist)
        fm=full_metrics(sub.y_true.values,sub.y_pred.values,y_naive=persist)
        rows.append(dict(track=track,horizon=h,tower=2,model="SARIMAX",order="varies(folds)",
            RMSE=round(r,3),MAE=round(mae,3),R2=(round(r2,3) if np.isfinite(r2) else np.nan),MBE=round(mbe,3),
            n_test=len(sub),skill_persist=round(1-r/rp,3) if rp>0 else np.nan,
            WAPE=round(fm["WAPE"],4) if np.isfinite(fm["WAPE"]) else np.nan,
            MASE=round(fm["MASE"],4) if np.isfinite(fm["MASE"]) else np.nan,
            sMAPE=round(fm["sMAPE"],4) if np.isfinite(fm["sMAPE"]) else np.nan,
            MAPE=round(fm["MAPE"],4) if np.isfinite(fm["MAPE"]) else np.nan,
            MAPE_n_excluded=fm["MAPE_n_excluded"]))
    return rows

ALL+=run_t2_folds("A",TA[2],EXOG_A,"h"); print("Track A Tower 2 folds done",flush=True)
ALL+=run_t2_folds("B",TB[2],EXOG_B,"D"); print("Track B Tower 2 folds done",flush=True)
R=pd.DataFrame(ALL); print("total rows",len(R))

  -- Track A Tower 2 fold (cut=2018-06-30): order search --


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


    order=(1,1,0) aic=159035.4 (2.7s)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


    order=(1,1,1) aic=156896.9 (9.5s)


...

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


    order=(2,1,1) aic=7584.8 (0.6s)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Track B Tower 2 folds done


total rows 27


## 4  Results -- SARIMAX vs B-03 (RF/XGB) and persistence

In [5]:
b03=pd.read_csv(RESULTS/"b03_summary.csv")
pd.set_option("display.width",200)
print("=== SARIMAX daily R2 (towers 4/9) ===")
print(R[(R.track=="B")&(R.tower.isin([4,9]))].pivot_table(index="tower",columns="horizon",values="R2").round(3).to_string())
print("\n=== SARIMAX hourly R2 (towers 4/9) ===")
print(R[(R.track=="A")&(R.tower.isin([4,9]))].pivot_table(index="tower",columns="horizon",values="R2").round(3).to_string())
print("\n=== B-03 RF/XGB daily R2 for comparison ===")
print(b03[(b03.track=="B")&(b03.tower.isin([4,9]))&(b03.model.isin(["RF","XGB"]))]
      .pivot_table(index=["tower","model"],columns="horizon",values="R2").round(3).to_string())
print("\n=== orders chosen (main split) ===")
print(R[["track","tower","order"]].drop_duplicates().to_string(index=False))
R.to_csv(RESULTS/"b03a_summary.csv",index=False); print("\nsaved results/b03a_summary.csv")

=== SARIMAX daily R2 (towers 4/9) ===
horizon     1      3      7      14
tower                              
4        0.371  0.066 -0.194 -0.330
9        0.282  0.055 -0.012 -0.023

=== SARIMAX hourly R2 (towers 4/9) ===
horizon     1      6      12     24     48
tower                                     
4       -0.168  0.147  0.038 -0.237 -0.215
9       -0.294  0.014  0.116 -0.679 -0.830

=== B-03 RF/XGB daily R2 for comparison ===
horizon         1      3      7      14
tower model                            
4     RF     0.357  0.345  0.287  0.270
      XGB    0.362  0.313  0.284  0.259
9     RF     0.388  0.350  0.364  0.342
      XGB    0.324  0.271  0.303  0.275

=== orders chosen (main split) ===
track  tower         order
    A      4     (2, 1, 1)
    A      9     (2, 1, 1)
    B      4     (2, 1, 1)
    B      9     (2, 1, 1)
    A      2 varies(folds)
    B      2 varies(folds)

saved results/b03a_summary.csv


## 5  Append to benchmarks.csv (B03a)

In [6]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="B03a"]
rows=[]
for _,r in R.iterrows():
    rows.append({"replication":"B03a","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":f"SARIMAX, order={r['order']}, SHAP-informed exogenous drivers (lsu_dens/WS/VPD/USTAR/PPFD+seasonality)",
        "track":r["track"],"horizon":int(r["horizon"]),
        "split":"fc_main" if r.tower in (4,9) else "fc_t2folds",
        "R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],"MBE":r["MBE"],"n_test":int(r["n_test"]),
        "skill_persist":r["skill_persist"],
        "WAPE":r["WAPE"],"MASE":r["MASE"],"sMAPE":r["sMAPE"],"MAPE":r["MAPE"],"MAPE_n_excluded":r["MAPE_n_excluded"],
        "date":today,"notes":"B03a classical SARIMAX baseline (fills D-05 ARIMA roster gap); solo per-tower (no pooling, methodological departure); walk-forward eval via append(refit=False); compare vs B03 RF/XGB"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} B03a rows. Total {len(comb)}.")

Wrote 27 B03a rows. Total 3692.
